In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load Data
try:
    df = pd.read_parquet("data/analysis_sample.parquet")
except Exception:
    df = pd.read_csv(r"C:\Users\ugochukwu.emenike\PycharmProjects\PythonProject\powerbi\analysis_sample.csv")

# 2. Target definition & leakage isolation
df = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP']).copy()
df['HIGH_BP_FLAG'] = np.where((df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0)

leakage_cols = ['SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIAGNOSED_HYPERTENSION', 'BP_CATEGORY', 'SAMPLE_ID', 'HIGH_BP_FLAG']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])
y = df['HIGH_BP_FLAG']

# 3. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. End-to-end Pipeline Setup
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

full_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit on training data
full_model.fit(X_train, y_train)

# 5. Save model to 'models/' directory
os.makedirs("models", exist_ok=True)
model_path = "models/hypertension_model.joblib"
joblib.dump(full_model, model_path)
print(f"Model saved successfully to {model_path}")

# 6. Reload model in fresh object to verify portability
loaded_model = joblib.load(model_path)
print("Model reloaded successfully.")

# 7. Predict on 3 test rows
sample_batch = X_test.head(3).copy()
sample_actual = y_test.head(3).values

preds = loaded_model.predict(sample_batch)
probs = loaded_model.predict_proba(sample_batch)[:, 1]

# Display results table
test_results = pd.DataFrame({
    'Actual_High_BP': sample_actual,
    'Predicted_Class': preds,
    'Predicted_Risk_Score': np.round(probs, 4)
})
print("\n--- Test Predictions on 3 Reloaded Rows ---")
display(test_results)

Model saved successfully to models/hypertension_model.joblib
Model reloaded successfully.

--- Test Predictions on 3 Reloaded Rows ---


,Actual_High_BP,Predicted_Class,Predicted_Risk_Score
0,1,1,0.6536
1,1,1,0.5414
2,0,0,0.4575
